In [0]:
import os, shutil, subprocess, sys, tempfile


In [0]:
VOLUME_LIBS = "/Workspace/Users/andrezluz19@gmail.com/DataEngCodes/Databricks_codes/codes"

In [0]:
PROJETO = os.getcwd()
assert os.path.exists(os.path.join(PROJETO, "pyproject.toml")), f"pyproject.toml não encontrado em {PROJETO}"
print("Projeto:", PROJETO)

In [0]:
# Copia o projeto para uma pasta temporária: o build cria build/ e *.egg-info,
# e assim nada disso aparece como alteração na Git folder.
tmp = tempfile.mkdtemp()
fonte = os.path.join(tmp, "src")
shutil.copytree(PROJETO, fonte, ignore=shutil.ignore_patterns(".git", "build", "dist", "*.egg-info", "__pycache__"))
saida_local = os.path.join(tmp, "dist")

In [0]:
comando = [sys.executable, "-m", "pip", "wheel", "--no-deps", "-w", saida_local, fonte]
try:
    subprocess.run(comando, check=True)
except subprocess.CalledProcessError:
    # Sem acesso ao PyPI para baixar o setuptools do build: usa o setuptools já instalado no ambiente
    print("[INFO] Build isolado falhou; tentando com o setuptools do ambiente (--no-build-isolation).")
    subprocess.run(comando[:4] + ["--no-build-isolation"] + comando[4:], check=True)


In [0]:
# pip install --upgrade pip

In [0]:
wheel = next(f for f in os.listdir(saida_local) if f.endswith(".whl"))
os.makedirs(VOLUME_LIBS, exist_ok=True)
destino = os.path.join(VOLUME_LIBS, wheel)
if os.path.exists(destino):
    print(f"[ALERTA] {wheel} já existia no Volume e foi sobrescrito. "
          "Aumente a versão no pyproject.toml para o cache do ambiente reinstalar o pacote.")
shutil.copy(os.path.join(saida_local, wheel), destino)
shutil.rmtree(tmp, ignore_errors=True)